This is my locally adapted runtime because of library conflicts with my 5060 gpu

```bash
# Creating a custom environment
uv venv .venv-gpu # with python 3.12
source .venv-gpu/bin/activate # activate the environment
uv pip install tf-nightly[and-cuda] pip ipykernel pandas keras # install the libraries 
```
update
this tensorflow version works: tf-nightly[and-cuda]==2.21.0.dev20251017


In [1]:
import os
import re

import keras # Used for defining an training the model.
import pandas as pd # Used for loading the dataset.
import tensorflow as tf # Used for shuffling the dataset.
from IPython.display import display, HTML


# The following line provides configuration for Keras.
keras.utils.set_random_seed(128)  # For Keras layers.

I0000 00:00:1781112136.691873   92077 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781112136.854665   92077 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781112139.509809   92077 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
# loading dataset

africa_galore = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore.json"
)
dataset = africa_galore["description"].values
print("Loaded dataset with", dataset.shape[0], "paragraphs.")

Loaded dataset with 232 paragraphs.


In [3]:
#  tokenization

class SimpleWordTokenizer:
    """A simple word tokenizer.

    The tokenizer splits the text sequence based on whitespace, using the
    `encode` method to convert the text into a sequence of indices and the
    `decode` method to convert indices back into text.

    The simple word tokenizer that can be initialized with a corpus or using a
    provided vocabulary list

    Typical usage example:

        corpus = "Hello there!"
        tokenizer = SimpleWordTokenizer(text)
        print(tokenizer.encode('Hello'))

    """

    # Define constants.
    UNKNOWN_TOKEN = "<UNK>"
    PAD_TOKEN = "<PAD>"

    def __init__(self, corpus: list[str], vocabulary: list[str] | None = None):
        """Initializes the tokenizer with texts in corpus or with a vocabulary.

        Args:
          corpus: Input text dataset.
          vocabulary: A pre-defined vocabulary. If None,
              the vocabulary is automatically inferred from the texts.
        """

        if vocabulary is None:
            # Build the vocabulary from scratch.
            if isinstance(corpus, str):
                corpus = [corpus]

            # Convert text sequence to tokens.
            tokens = []
            for text in corpus:
                for token in self.space_tokenize(text):
                    tokens.append(token)

            # Create a vocabulary comprising of unique tokens.
            vocabulary = self.build_vocabulary(tokens)

            # Add special unknown and pad tokens to the vocabulary list.
            self.vocabulary = (
                [self.PAD_TOKEN] + vocabulary + [self.UNKNOWN_TOKEN]
            )

        else:
            self.vocabulary = vocabulary

        # Size of vocabulary.
        self.vocabulary_size = len(self.vocabulary)

        # Create token-to-index and index-to-token mappings.
        self.token_to_index = {}
        self.index_to_token = {}
        # Loop through all tokens in the vocabulary. enumerate automatically
        # assigns a unique index to each token.
        for index, token in enumerate(self.vocabulary):
            self.token_to_index[token] = index
            self.index_to_token[index] = token

        # Map the special tokens to their IDs.
        self.pad_token_id = self.token_to_index[self.PAD_TOKEN]
        self.unknown_token_id = self.token_to_index[self.UNKNOWN_TOKEN]

    def space_tokenize(self, text: str) -> list[str]:
        """Splits a given text on whitespace into tokens.

        Args:
            text: Text to split on whitespace.

        Returns:
            List of tokens after splitting `text`.
        """

        # Use re.split such that multiple spaces are treated as a single
        # separator.
        return re.split(" +", text)

    def join_text(self, text_list: list[str]) -> str:
        """Combines a list of tokens into a single string.

        The combined tokens, as a single string, are separated by spaces in the
        string.

        Args:
            text_list: List of tokens to be joined.

        Returns:
            String with all tokens joined with a whitespace.

        """
        return " ".join(text_list)

    def build_vocabulary(self, tokens: list[str]) -> list[str]:
        """Create a vocabulary list from the list of tokens.

        Args:
            tokens: The list of tokens in the dataset.

        Returns:
            List of unique tokens (vocabulary) in the dataset.
        """
        return sorted(list(set(tokens)))

    def encode(self, text: str) -> list[int]:
        """Encodes a text sequence into a list of indices.

        Args:
            text: The input text to be encoded.

        Returns:
            A list of indices corresponding to the tokens in the input text.
        """

        # Convert tokens into indices.
        indices = []
        unk_index = self.token_to_index[self.UNKNOWN_TOKEN]
        for token in self.space_tokenize(text):
            token_index = self.token_to_index.get(token, unk_index)
            indices.append(token_index)

        return indices

    def decode(self, indices: int | list[int]) -> str:
        """Decodes a list (or single index) of integers back into tokens.

        Args:
            indices: A single index or a list of indices to be
                decoded into tokens.

        Returns:
            A string of decoded tokens corresponding to the input indices.
        """

        # If a single integer is passed, convert it into a list.
        if isinstance(indices, int):
            indices = [indices]

        # Map indices to tokens.
        tokens = []
        for index in indices:
            token = self.index_to_token.get(index, self.unknown_token_id)
            tokens.append(token)

        # Join the decoded tokens into a single string.
        return self.join_text(tokens)


# Initialize the tokenizer. This will build the tokenizer's vocabulary with
# all the tokens that appear in the dataset.
tokenizer = SimpleWordTokenizer(dataset)

# Translate all tokens to their corresponding IDs.
encoded_tokens = []
for text in dataset:
    # Split text into tokens and translate the tokens to token IDs.
    token_ids = tokenizer.encode(text)
    encoded_tokens.append(token_ids)

In [4]:
#  computing length of shortest and longest sentence
shortest_paragraph_length =len(min([encoded_token for encoded_token in encoded_tokens], key=len))

longest_paragraph_length = len(max([encoded_token for encoded_token in encoded_tokens], key=len))

print(f"Length of the shortest paragraph is:", shortest_paragraph_length)
print(f"Length of the longest paragraph is:", longest_paragraph_length)


Length of the shortest paragraph is: 26
Length of the longest paragraph is: 318


In [5]:
# padding the dataset
# set `max_length` for padding and truncating data.

max_length = 234 

if max_length <= 0:
    display(
        HTML(
            f"<h3>Error:</h3><p>Max length must be greater than 0. Please"
            f" increase <code>max_length</code>.</p><p></p>"
        )
    )

elif max_length > longest_paragraph_length:
    display(
        HTML(
            f"<h3>Error:</h3><p>The padding token <code>"
            f" {tokenizer.pad_token_id}</code> will be added to all"
            f" sequences - you probably don't want that. Please reduce"
            f" <code>max_length</code>.</p><p></p>"
        )
    )

else:
    if max_length < longest_paragraph_length:
        display(
            HTML(
                f"<p><strong>Note:</strong> The longest paragraph has"
                f" {longest_paragraph_length} tokens,"
                f" but <code>max_length</code> is set to {max_length}."
                f" Paragraphs longer than <code>max_length</code> will be"
                " truncated.</p><p></p>"
            )
        )

    padded_sequences = keras.preprocessing.sequence.pad_sequences(
        encoded_tokens,
        maxlen=max_length,
        padding="post",
        truncating="post",
        value=tokenizer.pad_token_id,
    )

    print("New length of first paragraph:", len(padded_sequences[0]), "\n")

    print(
        "Padding makes the length of all sequences the same as the specified"
        " `max_length`."
    )

    print(
        "Notice the padded token IDs {tokenizer.pad_token_id} appearing at the"
        f" end of the sequence.\n"
    )
    print("Padded tokens of first paragraph:\n", padded_sequences[0])

New length of first paragraph: 234 

Padding makes the length of all sequences the same as the specified `max_length`.
Notice the padded token IDs {tokenizer.pad_token_id} appearing at the end of the sequence.

Padded tokens of first paragraph:
 [ 814  511  985 5092 4802 5183 2800 1363 4792 2134 2856 4792 1584 5092
 2088  814 1134 3043 2922  912 2821  170 2623 4792 2023 3807 3576  912
 1653 3772 4792 2775 1244  912 4409 3280 1030 4792 1158 3049 1992  912
 1868 2486 2437  135 5189 3422  445 3388 2078 4849 4792 3407 2706 1259
 4692 2856 4839 5183 4792 4078  814 3406 4259 4849 2389 4831 2707  912
 3821 1829 3522 2134 1030 2955  185 1076 2707 3683 5143 1849 4343 1030
 1546 1446 4983 2856 4792 2876 4078  814 3406 5092 3366 4788 2968 2151
 2938 5092  912 1450 3522 3101  912 1672 4849 4793 4295 2721  912 5036
 2224 3522 4792 4437 3522  513    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    

In [6]:
# preparing input and target

# For each example, extract all tokens except the last one.
input_sequences = padded_sequences[:, :-1]
# For each example, extract all tokens except the first one.
target_sequences = padded_sequences[:, 1:]

print("First 10 token IDs in first input sequence:", input_sequences[0, :10])
print(
    "First 10 tokens in first input sequence:",
    tokenizer.decode(input_sequences[0, :10]),
)

print("\n")

print("First 10 token IDs in first target sequence:", target_sequences[0, :10])
print(
    "First 10 tokens in target sequence:",
    tokenizer.decode(target_sequences[0, :10])
)

First 10 token IDs in first input sequence: [ 814  511  985 5092 4802 5183 2800 1363 4792 2134]
First 10 tokens in first input sequence: The Lagos air was thick with humidity, but the energy


First 10 token IDs in first target sequence: [ 511  985 5092 4802 5183 2800 1363 4792 2134 2856]
First 10 tokens in target sequence: Lagos air was thick with humidity, but the energy in


In [23]:
# shuffling the dataset and specifying the batch size

# Create TensorFlow dataset to prepare sequences.
tf_dataset = tf.data.Dataset.from_tensor_slices((input_sequences, target_sequences))

# Randomly shuffle the dataset.
# The buffer_size determines how many examples from the dataset
# are held in memory before shuffling.
# If you are working with a very large dataset,
# reduce the buffer_size as needed.
tf_dataset = tf_dataset.shuffle(buffer_size=len(input_sequences))

# Specify batch size.
batch_size = 32  # @param {type: "number"}

# Create batches.
batches = tf_dataset.batch(batch_size)

for batch in batches.take(1):
    print(batch)

(<tf.Tensor: shape=(32, 233), dtype=int32, numpy=
array([[ 814,  153,  137, ...,    0,    0,    0],
       [ 871, 2932,  912, ...,    0,    0,    0],
       [ 360,  615,  662, ...,    0,    0,    0],
       ...,
       [ 323,  872, 2932, ...,    0,    0,    0],
       [ 808, 2932,  912, ...,    0,    0,    0],
       [ 221, 4445, 2932, ...,    0,    0,    0]],
      shape=(32, 233), dtype=int32)>, <tf.Tensor: shape=(32, 233), dtype=int32, numpy=
array([[ 153,  137,  985, ...,    0,    0,    0],
       [2932,  912, 3775, ...,    0,    0,    0],
       [ 615,  662, 2932, ...,    0,    0,    0],
       ...,
       [ 872, 2932,  912, ...,    0,    0,    0],
       [2932,  912, 2758, ...,    0,    0,    0],
       [4445, 2932,  912, ...,    0,    0,    0]],
      shape=(32, 233), dtype=int32)>)


In [26]:
# get total number of batches

total_batches = 0
for batch in batches:
    total_batches += 1
print("Total number of batches is:", total_batches)

Total number of batches is: 8


In [ ]:
# initializing the model

model = training.create_model(
    max_length=max_length,
    vocabulary_size=tokenizer.vocabulary_size,
    learning_rate=1e-4
)